# Automatic Differentiation

## `jax.grad` on Scalar Functions

In [1]:
import jax
import jax.numpy as jnp

In [2]:
def f(x):
    return jnp.sum(x**2)

In [3]:
grad_f = jax.grad(f)

x = jnp.array([1.0, 2.0, 3.0])
print(f"f(x): {f(x)}")
print(f"grad f(x): {grad_f(x)}")

f(x): 14.0
grad f(x): [2. 4. 6.]


In [4]:
jax.make_jaxpr(grad_f)(x)

{ lambda ; a:f32[3]. let
    b:f32[3] = integer_pow[y=2] a
    c:f32[3] = integer_pow[y=1] a
    d:f32[3] = mul 2.0:f32[] c
    _:f32[] = reduce_sum[axes=(0,) out_sharding=None] b
    e:f32[3] = broadcast_in_dim 1.0:f32[]
    f:f32[3] = mul e d
  in (f,) }

In [5]:
def g(x):
    return jnp.sin(x[0]) * jnp.exp(x[1])

In [6]:
x = jnp.array([0.5, 1.0])

In [7]:
grad_g = jax.grad(g)

print(f"grad g(x)[0]: {grad_g(x)[0]}")
print(f"Expected: {jnp.cos(x[0]) * jnp.exp(x[1])}")
print()
print(f"grad g(x)[1]: {grad_g(x)[1]}")
print(f"Expected: {jnp.sin(x[0]) * jnp.exp(x[1])}")

grad g(x)[0]: 2.38551664352417
Expected: 2.38551664352417

grad g(x)[1]: 1.3032137155532837
Expected: 1.3032137155532837


In [8]:
jax.make_jaxpr(grad_g)(x)

{ lambda ; a:f32[2]. let
    b:f32[1] = slice[limit_indices=(1,) start_indices=(0,) strides=None] a
    c:f32[] = squeeze[dimensions=(0,)] b
    d:f32[] = sin c
    e:f32[] = cos c
    f:f32[1] = slice[limit_indices=(2,) start_indices=(1,) strides=None] a
    g:f32[] = squeeze[dimensions=(0,)] f
    h:f32[] = exp g
    _:f32[] = mul d h
    i:f32[] = mul d 1.0:f32[]
    j:f32[] = mul 1.0:f32[] h
    k:f32[] = mul i h
    l:f32[1] = broadcast_in_dim k
    m:f32[2] = pad[padding_config=((1, np.int64(0), 0),)] l 0.0:f32[]
    n:f32[] = mul j e
    o:f32[1] = broadcast_in_dim n
    p:f32[2] = pad[padding_config=((0, np.int64(1), 0),)] o 0.0:f32[]
    q:f32[2] = add_any m p
  in (q,) }

## `value_and_grad`: The Training Loop Pattern

In [9]:
def loss(w, x, y_true):
    y_pred = jnp.dot(w, x)
    return jnp.mean((y_pred - y_true) ** 2)

In [10]:
w = jnp.array([1.0, 2.0, 3.0])
x = jnp.array([0.5, 1.0, 2.0])
y_true = jnp.array(5.0)

In [11]:
loss_val, grad_w = jax.value_and_grad(loss)(w, x, y_true)
print(f"Loss: {loss_val}")
print(f"Gradient: {grad_w}")

Loss: 12.25
Gradient: [ 3.5  7.  14. ]


In [12]:
loss_val, (grad_w, grad_x) = jax.value_and_grad(loss, argnums=(0, 1))(w, x, y_true)

In [13]:
print(f"Grad w.r.t. w: {grad_w}")
print(f"Grad w.r.t. x: {grad_x}")

Grad w.r.t. w: [ 3.5  7.  14. ]
Grad w.r.t. x: [ 7. 14. 21.]


## Gradients Through `Pytrees`: Parameters as Dicts

In [14]:
def mlp_loss(params, x, y_true):
    h = jnp.tanh(params["w1"] @ x + params["b1"])
    y_pred = params["w2"] @ h + params["b2"]
    return jnp.mean((y_pred - y_true) ** 2)

In [15]:
import jax.random as jrandom

key = jrandom.PRNGKey(0)
key, k1, k2 = jrandom.split(key, 3)
params = {
    "w1": jrandom.normal(k1, (4, 3)),
    "b1": jnp.zeros(4),
    "w2": jrandom.normal(k2, (1, 4)),
    "b2": jnp.zeros(1),
}

In [16]:
x = jnp.array([1.0, 2.0, 3.0])
y_true = jnp.array(1.0)
grads = jax.grad(mlp_loss)(params, x, y_true)

print(f"Grads: {grads}")

Grads: {'b1': Array([-0.00031471, -0.00086403,  0.00166988,  0.00229648], dtype=float32), 'b2': Array([-8.025627], dtype=float32), 'w1': Array([[-0.00031471, -0.00062943, -0.00094414],
       [-0.00086403, -0.00172806, -0.00259209],
       [ 0.00166988,  0.00333977,  0.00500965],
       [ 0.00229648,  0.00459295,  0.00688943]], dtype=float32), 'w2': Array([[ 8.025506,  8.025308,  8.023588, -8.024139]], dtype=float32)}


In [17]:
for k in grads:
    print(f"   grads['{k}'].shape = {grads[k].shape}")

   grads['b1'].shape = (4,)
   grads['b2'].shape = (1,)
   grads['w1'].shape = (4, 3)
   grads['w2'].shape = (1, 4)


## Higher-Order Derivatives

In [18]:
def f(x):
    return x**4

In [19]:
f1 = jax.grad(f)
f2 = jax.grad(f1)
f3 = jax.grad(f2)

In [20]:
x = jnp.array(2.0)
print(f"f(x): {f(x)}")
print(f"f'(x): {f1(x)}")
print(f"f''(x): {f2(x)}")
print(f"f'''(x): {f3(x)}")

f(x): 16.0
f'(x): 32.0
f''(x): 48.0
f'''(x): 48.0


In [21]:
jax.make_jaxpr(f1)(x)

{ lambda ; a:f32[]. let
    _:f32[] = integer_pow[y=4] a
    b:f32[] = integer_pow[y=3] a
    c:f32[] = mul 4.0:f32[] b
    d:f32[] = mul 1.0:f32[] c
  in (d,) }

In [22]:
jax.make_jaxpr(f2)(x)

{ lambda ; a:f32[]. let
    _:f32[] = integer_pow[y=4] a
    b:f32[] = integer_pow[y=3] a
    _:f32[] = mul 4.0:f32[] b
    c:f32[] = integer_pow[y=3] a
    d:f32[] = integer_pow[y=2] a
    e:f32[] = mul 3.0:f32[] d
    f:f32[] = mul 4.0:f32[] c
    _:f32[] = mul 1.0:f32[] f
    g:f32[] = mul 1.0:f32[] 1.0:f32[]
    h:f32[] = mul 4.0:f32[] g
    i:f32[] = mul h e
  in (i,) }

In [23]:
jax.make_jaxpr(f3)(x)

{ lambda ; a:f32[]. let
    _:f32[] = integer_pow[y=4] a
    b:f32[] = integer_pow[y=3] a
    _:f32[] = mul 4.0:f32[] b
    c:f32[] = integer_pow[y=3] a
    d:f32[] = integer_pow[y=2] a
    _:f32[] = mul 3.0:f32[] d
    _:f32[] = mul 4.0:f32[] c
    e:f32[] = integer_pow[y=3] a
    f:f32[] = integer_pow[y=2] a
    _:f32[] = mul 3.0:f32[] f
    g:f32[] = integer_pow[y=2] a
    h:f32[] = integer_pow[y=1] a
    i:f32[] = mul 2.0:f32[] h
    j:f32[] = mul 3.0:f32[] g
    k:f32[] = mul 4.0:f32[] e
    _:f32[] = mul 1.0:f32[] k
    l:f32[] = mul 1.0:f32[] 1.0:f32[]
    m:f32[] = mul 4.0:f32[] l
    _:f32[] = mul m j
    n:f32[] = mul m 1.0:f32[]
    o:f32[] = mul 3.0:f32[] n
    p:f32[] = mul o i
  in (p,) }

In [24]:
def h(x):
    return jnp.sin(x[0]) + x[1] ** 3


h1 = jax.grad(h)
h2 = jax.jacobian(jax.grad(h))

x = jnp.array([1.0, 2.0])
print(f"Gradient: {h1(x)}")
print(f"Hessian: {h2(x)}")

Gradient: [ 0.5403023 12.       ]
Hessian: [[-0.84147096  0.        ]
 [ 0.         12.        ]]


## Jacobians for Vector-Valued Functions

In [25]:
def f_vec(x):
    return jnp.array([x[0] ** 2, x[1] * x[0], jnp.sin(x[1])])

In [26]:
x = jnp.array([1.0, 2.0])
J = jax.jacobian(f_vec)(x)

In [27]:
print(f"Jacobian shape: {J.shape}")
print(f"Jacobian: {J}")

Jacobian shape: (3, 2)
Jacobian: [[ 2.          0.        ]
 [ 2.          1.        ]
 [ 0.         -0.41614684]]


In [28]:
# d(jnp.sin(x1))/dx1 = cos(x1)
print(f"J[2,1]: {J[2,1]}, expected: {jnp.cos(x[1])}")

J[2,1]: -0.416146844625473, expected: -0.416146844625473


## Hessians

In [29]:
def quadratic(x, A):
    return 0.5 * x @ A @ x

In [30]:
x = jnp.array([1.0, 2.0, 3.0])
A = jnp.array([[2.0, 0.5, 0.0], [0.5, 3.0, 1.0], [0.0, 1.0, 4.0]])

In [31]:
H = jax.hessian(quadratic)(x, A)
print(f"Hessian of A: {H}")

Hessian of A: [[2.  0.5 0. ]
 [0.5 3.  1. ]
 [0.  1.  4. ]]


## Forward-Mode Differentiation with `jvp`

In [32]:
def f(x):
    return jnp.array([jnp.sum(x**2), jnp.prod(x)])

In [33]:
x = jnp.array([1.0, 2.0, 3.0])
v = jnp.array([1.0, 0.0, 0.0])

In [34]:
primal_out, tangent_out = jax.jvp(f, (x,), (v,))
print(f"f(x): {primal_out}")
print(f"Jf(x) @ v: {tangent_out}")

f(x): [14.  6.]
Jf(x) @ v: [2. 6.]


In [35]:
def jacobian_fwd(f, x):
    n = x.shape[0]
    basis = jnp.eye(n)
    cols = [jax.jvp(f, (x,), (e,))[1] for e in basis]
    return jnp.stack(cols, axis=-1)

In [36]:
J_fwd = jacobian_fwd(f, x)
J_rev = jax.jacobian(f)(x)
jnp.allclose(J_fwd, J_rev)

Array(True, dtype=bool)

In [37]:
J_fwd

Array([[2., 4., 6.],
       [6., 3., 2.]], dtype=float32)

In [38]:
J_rev

Array([[2., 4., 6.],
       [6., 3., 2.]], dtype=float32)

## Differentiating Through JAX Control Flow

In [39]:
def abs_val(x):
    return jax.lax.cond(x > 0, lambda x: x, lambda x: -x, x)

In [40]:
x_pos, x_neg = jnp.array(2.0), jnp.array(-3.0)

In [41]:
print(f"grad(|x|) at x=2: {jax.grad(abs_val)(x_pos)}")
print(f"grad(|x|) at x=-3: {jax.grad(abs_val)(x_neg)}")

grad(|x|) at x=2: 1.0
grad(|x|) at x=-3: -1.0


In [42]:
jaxpr = jax.make_jaxpr(jax.grad(abs_val))(jnp.array(1.0))
print(jaxpr)

{ lambda ; a:f32[]. let
    b:bool[] = gt a 0.0:f32[]
    c:i32[] = convert_element_type[new_dtype=int32 weak_type=False] b
    _:f32[] = cond[
      branches=(
        { lambda ; d:f32[]. let e:f32[] = neg d in (e,) }
        { lambda ; f:f32[]. let  in (f,) }
      )
    ] c a
    g:f32[] = cond[
      branches=(
        { lambda ; h:f32[]. let i:f32[] = neg h in (i,) }
        { lambda ; j:f32[]. let  in (j,) }
      )
    ] c 1.0:f32[]
  in (g,) }


In [43]:
def power(x, n):
    def body(_, val):
        return val * x

    return jax.lax.fori_loop(0, n, body, 1.0)

In [44]:
print(f"x^3 at x=2: {power(jnp.array(2.0), 3)}")
print(f"d/dx x^3 at x=2: {jax.grad(power)(jnp.array(2.0), 3)}")

x^3 at x=2: 8.0
d/dx x^3 at x=2: 12.0


In [45]:
try:
    jax.jit(jax.grad(power))(jnp.array(2.0), 3)
except Exception as e:
    print(type(e).__name__)
    print(e)

ValueError
Reverse-mode differentiation does not work for lax.while_loop or lax.fori_loop with dynamic start/stop values. Try using lax.scan, or using fori_loop with static start/stop.


## Carrying State: `jax.lax.scan`

In [ ]:
def ema(prices, decay=0.9):
    def step(carry, x):
        # jax.debug.print("Value of carry at runtime: {}", carry)
        # jax.debug.print("Value of x at runtime: {}", x)
        new = decay * carry + (1 - decay) * x
        # jax.debug.print("Value of new at runtime: {}\n", new)
        return new, new

    final, path = jax.lax.scan(step, 0.0, prices)
    return final, path

In [64]:
prices = jnp.array([1.0, 2.0, 3.0, 4.0, 5.0])
final, path = ema(prices)
print(f"final: {final}")
print(f"path: {path}")

final: 1.3144099712371826
path: [0.1        0.29       0.561      0.90489995 1.31441   ]


## `stop_gradient`: Cutting the Flow on Purpose

In [67]:
def loss_with_stop(x):
    target = jax.lax.stop_gradient(2.0 * x)
    return jnp.sum((x - target) ** 2)


def loss_without(x):
    target = 2.0 * x
    return jnp.sum((x - target) ** 2)

In [68]:
x = jnp.array([1.0, 3.0])
print(f"with stop gradient: {jax.grad(loss_with_stop)(x)}")
print(f"without stop gradient: {jax.grad(loss_without)(x)}")

with stop gradient: [-2. -6.]
without stop gradient: [2. 6.]
